# Agent Tool Schema：从模糊函数到可评测退款接口

**面试问题：工具名称、描述、JSON Schema、错误语义、幂等和分层指标怎样设计？**

## 回答主线

先把真实请求和资源合同摆出来，用最简单的方案建立成本或正确性基线，再手写核心控制逻辑并展示完整事件、指标和失败修正。断言只在最后保护少量关键不变量，前面的可见输入、过程和结果才是学习主体。

## 真实案例

电商 Agent 需要在“查订单、估算退款、创建退款、查询物流”四个工具中选择并填参数。案例使用六条可读用户请求，对比模糊名称/描述的关键词基线与清晰 schema，手写参数验证器，分别计算工具选择、参数合法和可执行成功率；最后展示金额单位含糊与未知字段为何必须在调用前失败。

### 输入预览：工具目录与六条黄金请求

In [1]:
tools = {  # 定义四个具有真实副作用和参数合同的电商工具。
    "get_order": {"description": "按订单号读取订单状态和已付金额，不产生副作用", "required": {"order_id": str}, "optional": {}, "side_effect": False},  # 查询订单是只读工具。
    "quote_refund": {"description": "试算指定订单的可退金额，不创建退款", "required": {"order_id": str, "reason": str}, "optional": {}, "side_effect": False},  # 退款试算与正式退款必须分开。
    "create_refund": {"description": "在用户确认后创建人民币退款，需要订单号、分为单位金额和幂等键", "required": {"order_id": str, "amount_fen": int, "idempotency_key": str}, "optional": {"reason": str}, "side_effect": True},  # 正式退款必须精确定义单位和幂等键。
    "track_shipment": {"description": "按运单号查询最新物流节点", "required": {"tracking_no": str}, "optional": {}, "side_effect": False},  # 物流查询使用运单号而非订单号。
}  # 完成工具 schema registry。
golden = [  # 构造覆盖选择、参数和副作用确认的六条黄金请求。
    {"query": "查一下订单 A100 的付款状态", "tool": "get_order", "args": {"order_id": "A100"}},  # 订单状态查询。
    {"query": "订单 A101 破损，先算算能退多少", "tool": "quote_refund", "args": {"order_id": "A101", "reason": "破损"}},  # 只要求试算而非执行退款。
    {"query": "确认给 A102 退 88 元，操作号 op-7", "tool": "create_refund", "args": {"order_id": "A102", "amount_fen": 8800, "idempotency_key": "op-7", "reason": "用户确认"}},  # 已确认的正式退款。
    {"query": "运单 SF998 到哪了", "tool": "track_shipment", "args": {"tracking_no": "SF998"}},  # 物流查询。
    {"query": "A103 是否支持退款，先别操作", "tool": "quote_refund", "args": {"order_id": "A103", "reason": "咨询"}},  # 明确禁止副作用的退款咨询。
    {"query": "读取 A104 订单", "tool": "get_order", "args": {"order_id": "A104"}},  # 简短订单读取请求。
]  # 完成小型执行式评测集。
print("工具目录：")  # 输出名称、描述和副作用供学习者观察。
for name, spec in tools.items():  # 逐工具展示关键 schema 字段。
    print(name, "|", spec["description"], "| required=", list(spec["required"]), "| side_effect=", spec["side_effect"])  # 显示模型选择与控制面验证所需信息。
print("黄金请求数量：", len(golden))  # 展示后续分层评测的样本规模。

工具目录：
get_order | 按订单号读取订单状态和已付金额，不产生副作用 | required= ['order_id'] | side_effect= False
quote_refund | 试算指定订单的可退金额，不创建退款 | required= ['order_id', 'reason'] | side_effect= False
create_refund | 在用户确认后创建人民币退款，需要订单号、分为单位金额和幂等键 | required= ['order_id', 'amount_fen', 'idempotency_key'] | side_effect= True
track_shipment | 按运单号查询最新物流节点 | required= ['tracking_no'] | side_effect= False
黄金请求数量： 6


## Baseline 基线：只按一个关键词选择工具

In [2]:
def keyword_baseline(query):  # 实现没有否定和副作用语义的脆弱关键词路由器。
    if "退" in query:  # 任意出现退款词就选择正式退款工具。
        return "create_refund"  # 该规则会把试算和咨询误判成副作用动作。
    if "运单" in query or "物流" in query:  # 用物流关键词选择查询工具。
        return "track_shipment"  # 返回物流查询工具。
    return "get_order"  # 其余请求统一回退订单读取。

baseline_predictions = [keyword_baseline(case["query"]) for case in golden]  # 对六条黄金请求执行关键词基线。
baseline_accuracy = sum(prediction == case["tool"] for prediction, case in zip(baseline_predictions, golden)) / len(golden)  # 计算工具选择准确率。
print("关键词基线结果：")  # 输出逐样本选择对照。
for case, prediction in zip(golden, baseline_predictions):  # 逐条展示误选副作用工具的样本。
    print(f'{case["query"]:<28} predicted={prediction:<14} gold={case["tool"]}')  # 显示关键词无法区分 quote 与 create。
print(f"基线工具选择准确率={baseline_accuracy:.1%}")  # 汇总最简单方案的质量。

关键词基线结果：
查一下订单 A100 的付款状态             predicted=get_order      gold=get_order
订单 A101 破损，先算算能退多少           predicted=create_refund  gold=quote_refund
确认给 A102 退 88 元，操作号 op-7     predicted=create_refund  gold=create_refund
运单 SF998 到哪了                 predicted=track_shipment gold=track_shipment
A103 是否支持退款，先别操作             predicted=create_refund  gold=quote_refund
读取 A104 订单                   predicted=get_order      gold=get_order
基线工具选择准确率=66.7%


### 核心实现：意图特征与严格参数验证

In [3]:
def select_tool(query):  # 实现显式区分查询、试算、正式提交和否定语义的教学路由器。
    if "运单" in query or "物流" in query:  # 物流实体优先路由到运单查询。
        return "track_shipment"  # 返回只读物流工具。
    if "退" in query and any(word in query for word in ("先算", "是否", "先别", "咨询")):  # 识别退款试算或禁止操作语义。
        return "quote_refund"  # 返回没有副作用的试算工具。
    if "退" in query and any(word in query for word in ("确认", "操作号")):  # 只有确认和操作标识同时出现才选择正式退款。
        return "create_refund"  # 返回高风险副作用工具。
    return "get_order"  # 其余订单读取请求保持只读。

def validate_args(tool_name, arguments):  # 手写 additionalProperties=false 风格的参数验证器。
    spec = tools[tool_name]  # 读取目标工具的可信 schema。
    allowed = set(spec["required"]) | set(spec["optional"])  # 计算允许出现的全部字段。
    unknown = set(arguments) - allowed  # 找出模型幻觉或注入产生的未知字段。
    missing = set(spec["required"]) - set(arguments)  # 找出执行必需但缺失的字段。
    wrong_types = [name for name, expected in {**spec["required"], **spec["optional"]}.items() if name in arguments and not isinstance(arguments[name], expected)]  # 检查每个已提供字段的 Python 类型。
    if unknown or missing or wrong_types:  # 任一 schema 问题都必须在调用前失败。
        return {"ok": False, "unknown": sorted(unknown), "missing": sorted(missing), "wrong_types": wrong_types}  # 返回结构化校验错误供模型修正。
    return {"ok": True, "unknown": [], "missing": [], "wrong_types": []}  # 返回参数可以进入后续授权门禁。

predictions = [select_tool(case["query"]) for case in golden]  # 使用改进路由器选择工具。
validations = [validate_args(case["tool"], case["args"]) for case in golden]  # 对黄金参数执行严格 schema 验证。
print("改进路由与参数验证：")  # 输出逐样本可执行结果。
for case, prediction, validation in zip(golden, predictions, validations):  # 逐条展示选择、参数和 gold。
    print(f'{case["query"]:<28} selected={prediction:<14} args_ok={validation["ok"]}')  # 显示所有层次而非只报总准确率。

改进路由与参数验证：
查一下订单 A100 的付款状态             selected=get_order      args_ok=True
订单 A101 破损，先算算能退多少           selected=quote_refund   args_ok=True
确认给 A102 退 88 元，操作号 op-7     selected=create_refund  args_ok=True
运单 SF998 到哪了                 selected=track_shipment args_ok=True
A103 是否支持退款，先别操作             selected=quote_refund   args_ok=True
读取 A104 订单                   selected=get_order      args_ok=True


## 结果解读：选择、参数和执行必须分层计分

In [4]:
selection_accuracy = sum(prediction == case["tool"] for prediction, case in zip(predictions, golden)) / len(golden)  # 计算工具选择层准确率。
argument_validity = sum(validation["ok"] for validation in validations) / len(golden)  # 计算参数 schema 合法率。
executable_success = sum(prediction == case["tool"] and validation["ok"] for prediction, case, validation in zip(predictions, golden, validations)) / len(golden)  # 计算选择和参数同时正确的执行前成功率。
print("评测层次          关键词基线   改进方案")  # 输出分层指标表表头。
print(f"工具选择准确率      {baseline_accuracy:8.1%} {selection_accuracy:10.1%}")  # 对比模型是否选对能力。
print(f"参数 schema 合法率  {'未检查':>8} {argument_validity:10.1%}")  # 展示基线甚至没有参数门禁。
print(f"可执行前成功率      {'未定义':>8} {executable_success:10.1%}")  # 展示选择正确不等于可以执行。
print("解读：线上还要单独统计工具返回成功、后置条件成功和用户任务成功，不能用一个总分掩盖故障层次。")  # 给出生产评测口径。

评测层次          关键词基线   改进方案
工具选择准确率         66.7%     100.0%
参数 schema 合法率       未检查     100.0%
可执行前成功率           未定义     100.0%
解读：线上还要单独统计工具返回成功、后置条件成功和用户任务成功，不能用一个总分掩盖故障层次。


## 失败案例：金额单位含糊和未知 redirect_url

In [5]:
bad_arguments = {"order_id": "A102", "amount_fen": 88.0, "idempotency_key": "op-8", "redirect_url": "https://evil.example"}  # 构造元为单位浮点金额和未知外链字段。
bad_validation = validate_args("create_refund", bad_arguments)  # 在执行退款前进行确定性 schema 校验。
fixed_arguments = {"order_id": "A102", "amount_fen": 8800, "idempotency_key": "op-8", "reason": "用户确认"}  # 把 88 元转换为明确的 8800 分并删除未知字段。
fixed_validation = validate_args("create_refund", fixed_arguments)  # 验证修正后的参数。
print("错误参数：", bad_arguments)  # 展示类型和 additional property 两类风险。
print("错误校验结果：", bad_validation)  # 显示 unknown 与 wrong_types 的精确定位。
print("修正参数及结果：", fixed_arguments, fixed_validation)  # 展示模型或上游如何进行一次明确修订。

错误参数： {'order_id': 'A102', 'amount_fen': 88.0, 'idempotency_key': 'op-8', 'redirect_url': 'https://evil.example'}
错误校验结果： {'ok': False, 'unknown': ['redirect_url'], 'missing': [], 'wrong_types': ['amount_fen']}
修正参数及结果： {'order_id': 'A102', 'amount_fen': 8800, 'idempotency_key': 'op-8', 'reason': '用户确认'} {'ok': True, 'unknown': [], 'missing': [], 'wrong_types': []}


### 生产边界

In [6]:
schema_manifest = {name: {"required": list(spec["required"]), "optional": list(spec["optional"]), "side_effect": spec["side_effect"]} for name, spec in tools.items()}  # 构造可以版本化和回归测试的 schema manifest。
print("Schema manifest：", schema_manifest)  # 展示服务发布时应冻结的工具接口合同。
print("生产替换点：还需 JSON Schema、枚举/格式、对象级授权、审批、幂等存储、分页、错误分类、后置条件和真实工具 sandbox。")  # 明确教学类型检查与生产工具平台的差距。

Schema manifest： {'get_order': {'required': ['order_id'], 'optional': [], 'side_effect': False}, 'quote_refund': {'required': ['order_id', 'reason'], 'optional': [], 'side_effect': False}, 'create_refund': {'required': ['order_id', 'amount_fen', 'idempotency_key'], 'optional': ['reason'], 'side_effect': True}, 'track_shipment': {'required': ['tracking_no'], 'optional': [], 'side_effect': False}}
生产替换点：还需 JSON Schema、枚举/格式、对象级授权、审批、幂等存储、分页、错误分类、后置条件和真实工具 sandbox。


## 回归测试：只保护选择、安全参数和分层指标

In [7]:
assert selection_accuracy == 1.0  # 验证六条教学请求的工具意图全部正确区分。
assert predictions[1] == "quote_refund" and predictions[4] == "quote_refund"  # 验证试算和“先别操作”不会触发正式退款。
assert not bad_validation["ok"] and bad_validation["unknown"] == ["redirect_url"]  # 验证未知外链字段被 additionalProperties 门禁拒绝。
assert "amount_fen" in bad_validation["wrong_types"]  # 验证金额单位对应的整数类型错误可被定位。
assert fixed_validation["ok"] and executable_success == 1.0  # 验证修正参数可执行且分层评测结果一致。
print("回归测试通过：工具选择、无副作用咨询、未知字段、金额类型和执行前成功率均成立。")  # 用少量断言总结 schema 设计价值。

回归测试通过：工具选择、无副作用咨询、未知字段、金额类型和执行前成功率均成立。
